In [1]:
import numpy as np
from torch import nn
import torch

import xarray as xr
import matplotlib.pyplot as plt
import matplotlib
import cartopy
import rasterio

In [57]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(1, 1, 61),
            nn.ReLU(),
            nn.ConvTranspose2d(1, 1, 61),
            nn.ReLU(),
            nn.ConvTranspose2d(1, 1, 61),
            nn.ReLU(),
            nn.ConvTranspose2d(1, 1, 61),
            nn.ReLU(),
            nn.ConvTranspose2d(1, 1, 61),
            nn.ReLU(),
            nn.ConvTranspose2d(1, 1, 61)
        )

    def forward(self, x):
        return self.net(x)

# Data loadre

In [34]:
class ReKIS(torch.utils.data.Dataset):
    def __init__(self, Y):
        super().__init__()
        # obycajny crop
        Y = Y.isel(easting = slice(0, 400), northing = slice(0, 400))
        # stratia sa nazvy osi
        Y.rio.set_spatial_dims('easting', 'northing')

        # ziskanie trenovacich dat -> upscaling
        X = Y.rio.reproject(Y.rio.crs, resolution=(10_000, 10_000), resampling= rasterio.enums.Resampling.cubic_spline)

        # prevod do tenzorov
        self.X = torch.from_numpy(X.values).unsqueeze(1)
        self.Y = torch.from_numpy(Y.values).unsqueeze(1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]


class ReKISDataModule:
    """uchova loadre"""
    def __init__(self, batch_size, path):
        self.batch_size = batch_size
        self.path = path

    def setup(self):
        """vytvori ReKIS datasety z path"""
        Y = xr.open_mfdataset(self.path, decode_coords='all')
        Y = Y['TM']

        # rozdelenie na mnoziny a vytvorenie datasetov
        self.train = ReKIS(Y.sel(time=slice('1961', '1961')))
        self.val = ReKIS(Y.sel(time=slice('1962', '1962')))

    def train_dataloader(self):
        return torch.utils.data.DataLoader(self.train, batch_size = self.batch_size, shuffle = True)

    def val_dataloader(self):
        return torch.utils.data.DataLoader(self.val, batch_size = self.batch_size)

In [39]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [16]:
rekis_module = ReKISDataModule(32, f'../../rci_data/climate/ReKIS/KlimRefDS_v3.1_1961-2023/Raster/Tag/GK4/TM/*.nc')
rekis_module.setup()

train_loader = rekis_module.train_dataloader()
val_loader = rekis_module.val_dataloader()

In [58]:
net = SimpleCNN().to(device)

In [59]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(net.parameters(), lr = 0.001)

In [60]:
def train_epoch(model, loss_fn, optimizer, train_loader):
    running_cum_loss = 0.0
    for X, Y in train_loader:
        X = X.to(device)
        Y = Y.to(device)

        optimizer.zero_grad()
        output = model(X)
        loss = loss_fn(output, Y)
        loss.backward()

        optimizer.step()
        running_cum_loss += loss.item()
    return running_cum_loss / len(train_loader)

In [64]:
def eval_after_epoch(model, loss_fn, val_loader):
    running_cum_vloss = 0.0
    for X, Y in val_loader:
        X = X.to(device)
        Y = Y.to(device)

        with torch.no_grad():
            output = model(X)
            loss = loss_fn(output, Y)

        running_cum_vloss += loss.item()
    return running_cum_vloss / len(val_loader)

In [65]:
for i in range(10):
    print(f'Epoch {i + 1}: ')
    print(f'Train: {train_epoch(net, loss_fn, optimizer, train_loader)}')
    print(f'Val: {eval_after_epoch(net, loss_fn, val_loader)}')

Val: 103.12212411562602


In [78]:
model = net.to('cpu')

In [96]:
out = model(rekis_module.train[158][0]).detach().numpy()
print((out - out.mean()).sum()) # vsetky su rovnake

0.0035762787
